# DocuHelp: Intelligent Document Assistant  
### ITAI 2376 – Final Project  
**Student:** Jaret Sanchez  
**Environment:** Google Colab  
**Features:**  
- PDF ingestion  
- Text extraction  
- Chunking  
- Embeddings + FAISS retrieval  
- Topic classification (local ML model)  
- LangGraph agent for summarization  


# 1. Setup & Installation
This section installs all required libraries for DocuHelp and loads environment variables securely.


In [2]:
!pip install langchain langgraph faiss-cpu sentence-transformers pypdf python-dotenv openai transformers torch langchain-community langchain-huggingface langchain-openai

# 2. Load API Key
API keys are private and should not be visible in the notebook.

In [3]:
import os, getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

Enter your OpenAI API key: ··········


# 3. Upload PDF and Extract Text
Upload any PDF document. The agent will extract and process the text.

In [4]:
from google.colab import files
from langchain_community.document_loaders import PyPDFLoader

uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]

loader = PyPDFLoader(pdf_path)
pages = loader.load()

raw_text = "\n".join([p.page_content for p in pages])
raw_text[:1000]

Saving Climate_Change.pdf to Climate_Change.pdf


'READING MATERIAL\nRead About the Basics of Climate Change\nWHAT IS THE INTRO TO CLIMATE CHANGE?\nClimate is the average weather over many years. Earth’s average temperature has\nbeen increasing rapidly over the past century and a lot of evidence points to human\nactivity as the primary cause. Moving from a dependence on fossil fuels to renewable\nenergy resources can help decrease the amount of greenhouse gases humans put into\nthe air, and in turn, slow down the rate at which Earth’s global temperature is increasing.\nTo better understand climate change…\nLET’S BREAK IT DOWN!\nThe difference between weather and climate.\nWe often confuse weather and\nclimate when we are discussing\nclimate change. Weather is the day-\nto-day variations of the\natmosphere’s condition locally.\nWhich means we would describe the\ntemperature, sky conditions (e.g.,\nsunny, overcast), precipitation, and\nwhether it’s windy or calm. Generally,\nweather can only be predicted with some degree of accuracy for

# 4. Text Splitting
Long documents are split into overlapping chunks for better retrieval.

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=200
)

chunks = splitter.split_text(raw_text)
len(chunks)

15

# 5. Embeddings and Vector Store
We convert text chunks into embeddings and store them in FAISS for retrieval.

In [7]:
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

# Initialize the SentenceTransformer model
sentence_model = SentenceTransformer("all-MiniLM-L6-v2")

# Wrap the SentenceTransformer model with HuggingFaceEmbeddings
embedder = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2", model_kwargs={'device': 'cpu'})

docs = [Document(page_content=c) for c in chunks]

faiss_store = FAISS.from_documents(
    docs,
    embedding=embedder
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# 6. Topic Classification
A lightweight TF‑IDF + Logistic Regression classifier predicts the document's topic using the AG News dataset. This avoids Hugging Face authentication issues.


In [8]:
# Local AG News classifier
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Download AG News dataset (public CSV)
!wget -q https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/train.csv

# Load training data
train = pd.read_csv("train.csv", header=None)
X_train = train[1]
y_train = train[0] - 1  # labels 1–4 → 0–3

# Train TF-IDF + Logistic Regression classifier
vectorizer = TfidfVectorizer(stop_words="english", max_features=50000)
X_train_vec = vectorizer.fit_transform(X_train)

clf = LogisticRegression(max_iter=2000)
clf.fit(X_train_vec, y_train)

# Predict topic for the document
X_input = vectorizer.transform([raw_text[:1000]])
pred = clf.predict(X_input)[0]

topic_map = {
    0: "World News",
    1: "Sports",
    2: "Business",
    3: "Technology"
}

predicted_topic = topic_map[pred]
predicted_topic

'Technology'

# 7. LangGraph Agent
The agent retrieves relevant chunks and generates a summary using an LLM.

In [9]:
from typing import TypedDict, List

class AgentState(TypedDict):
    query: str
    retrieved_docs: List[str]
    summary: str

In [10]:
def retrieve_tool(state: AgentState):
    query = state["query"]
    results = faiss_store.similarity_search(query, k=4)
    state["retrieved_docs"] = [r.page_content for r in results]
    return state

In [11]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

def summarize_tool(state: AgentState):
    context = "\n\n".join(state["retrieved_docs"])
    prompt = f"Summarize the following document:\n\n{context}"
    response = llm.invoke(prompt)
    state["summary"] = response.content
    return state

In [12]:
from langgraph.graph import StateGraph, END

graph = StateGraph(AgentState)

graph.add_node("retrieve", retrieve_tool)
graph.add_node("summarize", summarize_tool)

graph.set_entry_point("retrieve")
graph.add_edge("retrieve", "summarize")
graph.add_edge("summarize", END)

agent = graph.compile()

# 8. Run the Agent
Ask the agent to summarize the document.

In [13]:
query = "Summarize the main ideas of this document."
result = agent.invoke({"query": query})
result["summary"]

"The document provides an introduction to climate change, explaining the significant difference between weather (short-term atmospheric conditions) and climate (long-term weather patterns). It underscores that Earth's temperature has been rising rapidly due to human activities, primarily the burning of fossil fuels, which increases greenhouse gases like carbon dioxide and methane. \n\nKey concepts include the greenhouse effect, which is essential for maintaining a habitable temperature on Earth, and the negative consequences of climate change, such as glacier melting, rising sea levels, increased forest fires, stronger hurricanes, and more severe droughts. The document also poses discussion questions about historical climate changes, the greenhouse effect's benefits, and potential natural and human-induced causes of climate change. \n\nOverall, the material emphasizes the importance of transitioning to renewable energy to mitigate greenhouse gas emissions and slow down global warming."

# 9. Final Output
Display the predicted topic and the generated summary.

In [14]:
print("Predicted Topic:", predicted_topic)
print("\nGenerated Summary:\n")
print(result["summary"])

Predicted Topic: Technology

Generated Summary:

The document provides an introduction to climate change, explaining the significant difference between weather (short-term atmospheric conditions) and climate (long-term weather patterns). It underscores that Earth's temperature has been rising rapidly due to human activities, primarily the burning of fossil fuels, which increases greenhouse gases like carbon dioxide and methane. 

Key concepts include the greenhouse effect, which is essential for maintaining a habitable temperature on Earth, and the negative consequences of climate change, such as glacier melting, rising sea levels, increased forest fires, stronger hurricanes, and more severe droughts. The document also poses discussion questions about historical climate changes, the greenhouse effect's benefits, and potential natural and human-induced causes of climate change. 

Overall, the material emphasizes the importance of transitioning to renewable energy to mitigate greenhouse 

# Reflection
Working on this project taught me how many different pieces have to come together to make an AI system actually work. I went in thinking it would be pretty straightforward, but once I started connecting the PDF loader, the text splitter, the embeddings, FAISS, the classifier, and the LangGraph agent, I realized how much coordination is involved. The biggest challenge was definitely the Hugging Face issues. No matter what I tried, the models wouldn’t load in Colab, so I had to switch to a local classifier. It was frustrating at first, but it ended up being a good lesson in adapting when something breaks and finding another way to reach the same goal.

Building the LangGraph agent was actually one of the more interesting parts. I liked seeing how the workflow becomes cleaner when you break it into nodes instead of writing one giant block of code. It made the whole process feel more organized and I could actually see how the state moves through the graph. Overall, this project helped me understand retrieval‑augmented systems in a more hands‑on way. I got more comfortable with embeddings, vector stores, and agent design, and I also learned how to troubleshoot when things don’t go the way you expect.
